# any-reduce-axis — ex1: row-wise any() to flag rows containing any True

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `any-reduce-axis`. Running the final beacon cell reports progress against the `Numpy: any() reduce along axis` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: any() reduce along axis` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`any-reduce-axis`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "any-reduce-axis"
DD_SUBTOPIC = "Numpy: any() reduce along axis"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Numpy: `any()` reduce along axis — quick refresher

`mask.any(dim=k)` collapses axis `k` of a bool tensor by OR-ing every element along it. Result has the same shape as `mask` with axis `k` removed:

```python
mask = torch.tensor([[False, False, True],
                     [False, False, False],
                     [True,  True,  False]])    # shape (3, 3)
mask.any(dim=1)   # tensor([ True, False,  True])   shape (3,)
mask.any(dim=0)   # tensor([ True,  True,  True])   shape (3,)
```

**Read it as 'at least one True along axis k'.** Row 0 has a True → True. Row 1 is all False → False. Row 2 has a True → True. `dim=0` collapses ROWS (reduces down the columns); `dim=1` collapses COLUMNS (reduces across each row).

**`numpy.any(arr, axis=k)` is the equivalent.** Same semantics, different keyword name (`axis=` vs `dim=`).

**Use `keepdim=True` to preserve rank.** `mask.any(dim=1, keepdim=True)` returns `(3, 1)` instead of `(3,)` — lets you broadcast the reduce result back against the original.

**`.all(dim=k)` is the AND-cousin.** True iff every element along axis `k` is True. Use it for 'no failures' checks; use `.any()` for 'at least one hit' checks.

### Exercise 1 — row-wise any() to flag rows containing any True

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `.any(dim=k)` to collapse one axis of a 2-D bool tensor, distinguishing `dim=0` (per-column) from `dim=1` (per-row) collapses.
> Keywords: any, reduce, axis, bool-mask
> ```

**KCs targeted:** `any-reduce-axis-collapse`, `axis-direction-convention`

Implement `ex1_row_has_true(mask)`. Take an `(N, M)` boolean tensor and return an `(N,)` boolean tensor where entry `i` is `True` iff row `i` of `mask` contains AT LEAST ONE `True`.

Inputs:
- `mask`: `(N, M)` bool tensor.

Output: `(N,)` bool tensor.

Constraints:
- Use `.any(dim=...)` — do NOT write a Python loop.
- Output dtype must be `torch.bool`.
- Output shape must be `(N,)` (NOT `(N, 1)`); do not pass `keepdim=True`.

In [ ]:
def ex1_row_has_true(mask: Tensor) -> Tensor:
    """(N, M) bool -> (N,) bool: True iff row has at least one True."""
    raise NotImplementedError()


def _test_ex1():
    # === Hand-traced reference ===
    mask = t.tensor([
        [False, False, True ],   # row 0: has True
        [False, False, False],   # row 1: all False
        [True,  True,  False],   # row 2: has True
        [False, True,  False],   # row 3: has True
    ])
    out = ex1_row_has_true(mask)
    expected = t.tensor([True, False, True, True])
    assert out.dtype == t.bool, f'expected bool, got {out.dtype}'
    assert out.shape == (4,), f'expected shape (4,), got {tuple(out.shape)}'
    assert t.equal(out, expected), f'expected {expected}, got {out}'

    # === All-True input -> all True ===
    all_true = t.ones(5, 3, dtype=t.bool)
    assert t.equal(ex1_row_has_true(all_true), t.ones(5, dtype=t.bool))

    # === All-False input -> all False ===
    all_false = t.zeros(5, 3, dtype=t.bool)
    assert t.equal(ex1_row_has_true(all_false), t.zeros(5, dtype=t.bool))

    # === Single-column input -> just that column's values ===
    col = t.tensor([[True], [False], [True]])
    assert t.equal(ex1_row_has_true(col), t.tensor([True, False, True]))

    # === The axis convention check ===
    # If a student used dim=0 by mistake on a (3, 5) input, they would
    # get a length-5 vector. The output length must equal N (the first dim).
    rect = t.tensor([
        [True,  False, False, False, False],   # row 0: has True
        [False, False, False, False, False],   # row 1: all False
        [False, False, False, False, True ],   # row 2: has True
    ])
    out_rect = ex1_row_has_true(rect)
    assert out_rect.shape == (3,), (
        f'output should have length N=3 (first dim), got {tuple(out_rect.shape)} '
        '— did you use dim=0 by mistake?'
    )
    assert t.equal(out_rect, t.tensor([True, False, True]))

    # === Larger random check vs reference Python loop ===
    rng = t.Generator().manual_seed(0)
    big = t.randint(0, 2, (32, 64), generator=rng).bool()
    ref = t.tensor([row.any().item() for row in big])
    assert t.equal(ex1_row_has_true(big), ref)
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_row_has_true(mask):
    return mask.any(dim=1)
```

**One line, one axis.** `mask.any(dim=1)` collapses axis 1 (the columns) and returns one boolean per row.

**Why not `dim=0`.** `dim=0` collapses ROWS, giving you ONE boolean per column. That's the opposite question — 'does this column contain any True'.

**Why not `dim=-1`.** It works (last axis == columns for 2-D), but `dim=1` is more explicit when you know the tensor is 2-D. Use `dim=-1` for code that works across arbitrary ranks.

**Generalizes.** `.all()`, `.sum()`, `.max()`, `.min()`, `.mean()` all share the same `dim=` API — same collapse rules, different reduction op.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()